In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

In [3]:
#load dataset

df = pd.read_csv("../data/combined_cleaned_backup_initial.csv")  # Use the original dataset before TDA


C:\Users\KARISHMA\AppData\Local\Temp\ipykernel_12616\3307713690.py:3: DtypeWarning: Columns (2,5,6,9,12,13,14,15,16,18,21,24,25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/combined_cleaned_backup_initial.csv")  # Use the original dataset before TDA


In [4]:
#Dropping unnecessary columns

columns_to_keep = [
    'customer_id',
    'gender',
    'age',
    'tenure',
    'monthlycharges',
    'paymentmethod',
    'churn',
    'industry']

df = df[columns_to_keep]

In [5]:
#feature engineering

# A: Tenure groups
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 6, 24, 60],
    labels=['short', 'medium', 'long']
)

df['is_new_customer'] = (df['tenure'] < 3).astype(int)
df['is_loyal_customer'] = (df['tenure'] > 24).astype(int)

# B: Spending features
df['spend_per_tenure'] = df['monthlycharges'] * df['tenure']
df['high_spender'] = (df['monthlycharges'] > df['monthlycharges'].median()).astype(int)

# C: Risky payment methods
df['paymentmethod_risky'] = df['paymentmethod'].apply(lambda x: 1 if "electronic" in str(x).lower() else 0)

# D: Age groups
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 45, 65, 100],
                         labels=['young', 'adult', 'mid_age', 'senior'])


In [6]:
#splitting into X and y

X = df.drop(['customer_id', 'churn'], axis=1)
y = df['churn']

In [7]:
#train - test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
#preprocessing
numeric_features = ['age', 'tenure', 'monthlycharges', 'spend_per_tenure']
categorical_features = ['gender', 'paymentmethod', 'industry', 'tenure_group', 'age_group']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [11]:
#pipeline with Smote and XGBoost
model = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('clf', XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_estimators=250,
        learning_rate=0.1,
        max_depth=6
    ))
])

In [12]:
#training and predicting
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [13]:
#metrics

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.6751064836725036
Precision: 0.5884194053208138
Recall: 0.6588785046728972
F1 Score: 0.6216588591898594
Confusion Matrix:
 [[1725  789]
 [ 584 1128]]
